# `train.csv` Has 4,407 Studies and 58 Labels

**`df.dropna()` leaves you with 58 rows. The other 4,349 studies aren't missing data —
their labels are in the report text.**

This notebook counts what's actually in `train.csv`. That's all it does. No model, no
extraction method, no strategy — but the count changes what kind of problem this is, and
it takes one `value_counts()` to see.

Four things it establishes:

1. **58 of 4,407 studies (1.3%) carry labels.** Not partially — those 58 have all twelve
   filled, and the rest have all twelve empty. It's all-or-nothing per study.
2. **Every study has a report**, including all 4,349 unlabelled ones. The diagnostic
   information is present; it just isn't in the label columns.
3. **The corpus is multi-script** — 541 reports (12.3%) contain Greek or Cyrillic — and the
   file has more lines than rows because report text contains newlines, so a naive line
   count overstates the dataset by 13×.
4. **New in v2, and it partly corrects this notebook:** section 3 warned that a
   Latin-script text pipeline fails *silently* on those 541 reports. That was an argument,
   never a measurement. Section 4 measures it against a published LLM labelling that
   encodes refusals explicitly — and **the refusals go the other way.** 25.9% of label
   cells are refused on Latin-only reports against 21.9% on non-Latin ones, a gap no
   shuffle out of 2,000 reached.

In [1]:
import csv, collections, re, pathlib

def _safe_roots():
    """Search roots for shipped data. Deliberately excludes /, home and the
    system temp dirs: rglob on those crawls the whole filesystem, which turns a
    missing-data case into a hang (or an OSError on macOS AppTranslocation)."""
    import os
    here = pathlib.Path.cwd().resolve()
    bad = {pathlib.Path("/"), pathlib.Path.home(), pathlib.Path("/tmp"),
           pathlib.Path("/private"), pathlib.Path("/private/tmp"), pathlib.Path("/var")}
    roots = [pathlib.Path("/kaggle/input"), here, *list(here.parents)[:4]]
    return [r for r in roots if r.exists() and r not in bad]

import matplotlib.pyplot as plt
import numpy as np

def find_train():
    for root in _safe_roots():
        for p in root.rglob("train.csv"):
            try: head = p.open(encoding="utf-8", errors="ignore").readline()
            except Exception: continue
            # Validate it's THIS competition's file -- several ship a train.csv.
            if "StudyInstanceUID" in head and "Report" in head:
                return p
    return None

path = find_train()
if path is None:
    raise FileNotFoundError("Attach the RSNA knee competition data.")
print(f"reading {path}")

with path.open(encoding="utf-8") as fh:
    n_lines = sum(1 for _ in fh)
rows = list(csv.DictReader(path.open(encoding="utf-8")))
LABELS = [c for c in rows[0] if c not in ("StudyInstanceUID", "Report")]

print(f"\nlines in file : {n_lines:,}")
print(f"parsed studies: {len(rows):,}")
print(f"ratio         : {n_lines/len(rows):.1f} lines per study  <- reports contain newlines")
print(f"\nlabel columns ({len(LABELS)}): {', '.join(LABELS)}")

## 1. How many studies actually carry labels

The label columns are empty strings, not zeros — which matters, because an empty string is
"unknown", while a zero is a negative finding. Conflating them silently invents ~4,349
negative examples per class that nobody asserted.

In [2]:
def is_labelled(r):  return any(r[l] != "" for l in LABELS)

labelled   = [r for r in rows if is_labelled(r)]
unlabelled = [r for r in rows if not is_labelled(r)]

print(f"total studies       {len(rows):>6,}")
print(f"labelled            {len(labelled):>6,}   {100*len(labelled)/len(rows):>5.1f}%")
print(f"unlabelled          {len(unlabelled):>6,}   {100*len(unlabelled)/len(rows):>5.1f}%")

# All-or-nothing, or partially filled?
filled = collections.Counter(sum(1 for l in LABELS if r[l] != "") for r in labelled)
print(f"\nlabel columns filled per labelled study: {dict(sorted(filled.items()))}")
print("-> labelling is all-or-nothing per study, never partial.")

have_report = sum(1 for r in rows if r["Report"].strip())
print(f"\nstudies with a report: {have_report:,} ({100*have_report/len(rows):.1f}%)")
print("Every unlabelled study still has its report text.")

## 2. What the 58 labelled studies look like

Worth knowing before treating them as a validation set: they are **not** a random sample.
Prevalence in them is far higher than any knee-MRI population would show, which is what
you'd expect from a deliberately enriched annotation set.

In [3]:
prev = {}
for l in LABELS:
    vals = [r[l] for r in labelled if r[l] != ""]
    pos = sum(1 for v in vals if v == "1")
    prev[l] = 100*pos/len(vals)

order = sorted(prev, key=prev.get, reverse=True)
print(f"{'finding':<20} {'positive':>9} {'of':>4} {'rate':>7}")
print("-"*44)
for l in order:
    vals = [r[l] for r in labelled if r[l] != ""]
    pos = sum(1 for v in vals if v == "1")
    print(f"{l:<20} {pos:>9} {len(vals):>4} {prev[l]:>6.1f}%")

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.barh(order[::-1], [prev[l] for l in order[::-1]], color="#0074D9")
ax.set_xlabel("% positive among the 58 labelled studies")
ax.set_title("Prevalence in the labelled subset (not population prevalence)", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

n_pos = [sum(1 for l in LABELS if r[l]=="1") for r in labelled]
print(f"\nfindings per labelled study: mean {np.mean(n_pos):.1f}, max {max(n_pos)}")
print("Multi-label and heavily co-occurring -- these are not mutually exclusive classes.")

## 3. The corpus is multi-script, and my first pass at this was wrong

**Correcting v1 of this notebook.** It classified the reports with a keyword heuristic
that tested only English and Spanish terms, and bucketed everything else as "ambiguous".
That heuristic was structurally incapable of finding a third language, so the ambiguous
bucket was not genuinely undecidable — it was hiding real languages, and it was large.
The notebook `RSNA Knee: read the report, then the knee` by prvsiyan mentions Greek
reports, which is what tipped me off; I had not considered Greek at all.

Redone by Unicode script instead of keywords. A report counts as containing a script if
it has three or more consecutive letters of it, and a report can contain more than one:

In [4]:
# "3+ consecutive letters of the script" -- deliberately strict, so a stray symbol
# (a lone mu, a degree sign) does not classify a report as Greek.
RUN = {
    "Latin":    re.compile(r"[A-Za-zÀ-ɏ]{3,}"),
    "Greek":    re.compile(r"[Ͱ-Ͽἀ-῿]{3,}"),
    "Cyrillic": re.compile(r"[Ѐ-ӿ]{3,}"),
}

script_of = {name: [bool(rx.search(r["Report"])) for r in rows] for name, rx in RUN.items()}
non_latin = [g or c for g, c in zip(script_of["Greek"], script_of["Cyrillic"])]

for name in ("Latin", "Greek", "Cyrillic"):
    n = sum(script_of[name])
    print(f"  {name:<9} {n:>5,}  {100*n/len(rows):>5.1f}%")
print(f"  {'non-Latin':<9} {sum(non_latin):>5,}  {100*sum(non_latin)/len(rows):>5.1f}%   <- Greek or Cyrillic present")

# The Greek is prose, not stray symbols: check that every Greek-flagged report qualifies.
assert all(RUN["Greek"].search(r["Report"]) for r, g in zip(rows, script_of["Greek"]) if g)
print("\nOne Greek report opening, for illustration:")
greek_ex = next(r for r, g in zip(rows, script_of["Greek"]) if g)
print(" ", greek_ex["Report"][:90].replace("\n", " "))

lens = sorted(len(r["Report"]) for r in rows)
print(f"\nreport length (chars): median {lens[len(lens)//2]:,}  p90 {lens[int(.9*len(lens))]:,}  max {lens[-1]:,}")
print("\nfirst report, truncated:")
print(" ", rows[0]["Report"][:220].replace("\n", " "))

This is a script check, not language identification — Latin covers English, Spanish and
anything else in that alphabet — so treat it as a lower bound on the diversity present.

One codepoint trap while you are here: **U+03BC GREEK SMALL LETTER MU and U+00B5 MICRO
SIGN render identically and do not match each other.** A regex for one silently misses
the other.

## 4. Does a text pipeline go quiet on those reports? Measured — and it does not

Section 3 is a warning as it stands: a pipeline built on Latin-script terms does not fail
loudly on the 541 non-Latin reports, it extracts nothing and returns a confident negative.
Same shape as the empty-versus-zero trap in section 1 — a silent wrong answer rather than
an error.

**That was an argument, never a measurement, and it is now testable.** stevenleehans
published an LLM labelling of all 4,407 reports
([dataset](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels),
[discussion](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/discussion/733932))
which encodes *"the report does not address this"* as an explicit **0.5** rather than
guessing. That sentinel is a refusal counter. If a text pipeline were quietly failing on
the non-Latin reports, its refusals should pile up there.

Reproducing both source numbers first, as hard assertions — if either fails, the join is
against the wrong file and nothing below means anything:

In [5]:
def find_llm_labels():
    # Exact filename: this dataset also ships llm_labels_v2.csv and
    # llm_labels_v4_blend.csv, which are imputed and would give different numbers.
    for root in _safe_roots():
        for p in root.rglob("llm_labels_full.csv"):
            return p
    return None

lpath = find_llm_labels()
if lpath is None:
    raise FileNotFoundError(
        "Attach the dataset stevenleehans/rsna-knee-llm-report-labels (llm_labels_full.csv).")
print(f"reading {lpath}")

lrows = list(csv.DictReader(lpath.open(encoding="utf-8")))
LFIND = [c for c in lrows[0] if c != "StudyInstanceUID"]
assert set(LFIND) == set(LABELS), "label columns differ between the two files"

SENTINEL = 0.5
cells = [[float(r[c]) for c in LFIND] for r in lrows]
n_cells = sum(len(row) for row in cells)
n_sent  = sum(1 for row in cells for v in row if v == SENTINEL)

print(f"\nCHECK 1 - refusal sentinel (published as 25.4% of all label cells)")
print(f"  cells at exactly 0.5: {n_sent:,} / {n_cells:,} = {100*n_sent/n_cells:.1f}%")
assert n_sent == 13438 and n_cells == 52884, "sentinel count does not reproduce -- wrong file?"

print(f"\nCHECK 2 - script counts from section 3 above")
print(f"  Greek {sum(script_of['Greek']):,} / Cyrillic {sum(script_of['Cyrillic']):,} "
      f"/ non-Latin {sum(non_latin):,}")
assert sum(script_of["Greek"]) == 321 and sum(script_of["Cyrillic"]) == 220

by_uid = {r["StudyInstanceUID"]: r for r in lrows}
matched = [(r, by_uid[r["StudyInstanceUID"]]) for r in rows if r["StudyInstanceUID"] in by_uid]
print(f"\nCHECK 3 - join coverage")
print(f"  matched {len(matched):,} / {len(rows):,} studies ({100*len(matched)/len(rows):.1f}%)")
assert len(matched) == len(rows), "incomplete join -- the groups below would not be comparable"

Now the cross-tab. Every study carries all twelve finding cells, so the *finding mix is
identical across the two groups by construction* — there is no composition confound to
argue about and the two rates compare directly.

In [6]:
groups = {"Latin only": [], "non-Latin present": []}
for (t, l), nl in zip(matched, non_latin):
    groups["non-Latin present" if nl else "Latin only"].append(l)

def refusal(rs):
    vals = [float(r[c]) for r in rs for c in LFIND]
    return sum(1 for v in vals if v == SENTINEL) / len(vals), len(vals)

rates = {}
for g, rs in groups.items():
    rate, n = refusal(rs)
    rates[g] = rate
    print(f"  {g:<20} {len(rs):>5,} studies   refused {100*rate:>5.1f}% of {n:,} cells")

gap = rates["non-Latin present"] - rates["Latin only"]
print(f"\n  ratio (non-Latin / Latin-only): {rates['non-Latin present']/rates['Latin only']:.3f}x")
print(f"  absolute difference           : {100*gap:+.1f} pp")
print("\n  -> It refuses LESS on the non-Latin reports, not more.")

print("\nSplit by individual script (the groups overlap -- a report can contain both):")
for name in ("Greek", "Cyrillic"):
    rs = [l for (t, l), f in zip(matched, script_of[name]) if f]
    print(f"  {name:<9} {len(rs):>5,} studies   refused {100*refusal(rs)[0]:>5.1f}%")

### Is that bigger than noise, and is it just report length?

Two checks. The permutation test shuffles the script label across all 4,407 studies and
recomputes the gap, which is the null "script has nothing to do with refusal". The length
check is the mundane explanation — a longer report gives a labeller more to work with.

In [7]:
rng = np.random.default_rng(0)
flat = np.array([[float(l[c]) == SENTINEL for c in LFIND] for t, l in matched])
mask = np.array(non_latin)
obs  = flat[mask].mean() - flat[~mask].mean()

idx  = np.arange(len(flat))
null = np.empty(2000)
for i in range(2000):
    rng.shuffle(idx)
    m = np.zeros(len(flat), dtype=bool); m[idx[:mask.sum()]] = True
    null[i] = flat[m].mean() - flat[~m].mean()

print(f"  observed difference {100*obs:+.2f} pp")
print(f"  null sd             {100*null.std():.2f} pp")
print(f"  shuffles reaching it: {int((np.abs(null) >= abs(obs)).sum())} / 2,000")

print("\nReport length by script group:")
for g, sel in (("Latin only", ~mask), ("non-Latin present", mask)):
    ls = np.array([len(t["Report"]) for t, l in matched])[sel]
    print(f"  {g:<20} median {int(np.median(ls)):>5,} chars   mean {ls.mean():>7,.0f}")
print("\n  -> The non-Latin reports are SHORTER, which predicts MORE refusals, not fewer.")
print("     Length runs against this result rather than explaining it.")

### Per finding — and the two columns that go the other way

Ten of the twelve findings are refused less often on non-Latin reports. Two are refused
*more*, and they are already the two most-refused columns:

In [8]:
per = []
for c in LFIND:
    a = np.mean([float(l[c]) == SENTINEL for (t, l), f in zip(matched, non_latin) if not f])
    b = np.mean([float(l[c]) == SENTINEL for (t, l), f in zip(matched, non_latin) if f])
    per.append((c, a, b, b - a))
per.sort(key=lambda r: r[3])

for c, a, b, d in per:
    print(f"  {c:<18} Latin {100*a:>5.1f}%   non-Latin {100*b:>5.1f}%   gap {100*d:>+5.1f} pp")

fig, ax = plt.subplots(figsize=(9.5, 5.0))
names = [c for c, a, b, d in per]
gaps  = [100*d for c, a, b, d in per]
ax.barh(names, gaps, color=["#D62728" if g > 0 else "#0074D9" for g in gaps])
ax.axvline(0, color="black", lw=1)
ax.axvline(100*null.std()*2, color="grey", ls="--", lw=1)
ax.axvline(-100*null.std()*2, color="grey", ls="--", lw=1)
ax.set_xlabel("refusal rate on non-Latin reports minus Latin-only (percentage points)")
ax.set_title("Blue = refused less on non-Latin reports. Red = refused more.\n"
             "Dashed lines: ±2 permutation SD of the overall gap", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**The caveat is the actionable half, so please don't take only the headline.**

A lower refusal rate is **not** evidence of a correct read. A labeller that reads Greek
fluently and one that hallucinates fluently produce the identical signature here: fewer
refusals. Refusal rate measures willingness to answer, not accuracy.

And the accuracy question is not currently answerable, because the only ruler is the
annotator-labelled studies:

In [9]:
gold_non_latin = sum(1 for r, nl in zip(rows, non_latin) if is_labelled(r) and nl)
print(f"  gold-labelled studies       : {len(labelled):>3,}")
print(f"  of which non-Latin script   : {gold_non_latin:>3,}")
print("\n  -> That is the entire ruler available for checking accuracy on the 541")
print("     non-Latin studies. It is not enough to settle it either way.")

So the claim section 4 actually supports, and no more than this: **the specific failure
mode section 3 warns about — a pipeline going quiet on non-Latin reports — does not appear
in this label set.** Whether the answers it gives on those 541 studies are *right* is a
separate question, and 6 gold studies cannot settle it.

One thing that may matter more than the headline, for anyone reading topic 733932: its
section 4 turns on how differently Synovitis and Baker's behave when a report is silent.
Those are exactly the two columns whose silence rate shifts by script above, so that
argument is being measured across a mixed population — worth checking whether it holds
within script group.

## What this changes

With 58 labelled studies across 12 correlated findings, there is not enough supervision to
train a twelve-class imaging model directly, and not enough to validate one either — a
held-out split of 58 gives single-digit positives for the rarer findings.

What there *is*: **4,407 free-text reports covering every study, including the 4,349
without labels.** The diagnostic information exists for the whole training set; it's in
prose rather than in columns.

I'm deliberately not proposing how to use that — this notebook is a count, not a method.
But the count is worth having before choosing an approach, because the obvious first move
(`dropna()`, then train) quietly discards 98.7% of the available signal.

Three smaller traps in the same file:

- **Empty is not zero.** Filling blanks with 0 invents thousands of unasserted negatives.
- **58,556 lines, 4,407 studies.** Reports contain newlines, so `wc -l` overstates the
  dataset 13×. Parse with a real CSV reader.
- **U+03BC and U+00B5** both render as µ and do not match each other.

And one thing section 4 changes about the report text specifically: if your pipeline drops
or down-weights reports it cannot parse, you are discarding 541 studies (12.3%) that at
least one published labeller did *not* choke on. Whether it read them *correctly* is
unsettled and, with 6 non-Latin studies among the 58 gold, currently unsettleable.

---

*Every number computed in-cell from `train.csv` and `llm_labels_full.csv`. The file lookup
validates the header before using it, because several competitions in a shared workspace
ship a `train.csv`, and the three reproduction checks in section 4 are `assert`s rather
than prints so a wrong file stops the notebook instead of quietly changing the answer.*

*v2 (2026-08-15) replaces the keyword language heuristic with a Unicode-script count and
adds section 4. The refusal analysis was posted to topic 734055 the same day; credit to
stevenleehans for publishing a label set with an explicit refusal code, which is the only
reason any of section 4 is measurable, and to prvsiyan for the Greek observation.*

*Corrections welcome — particularly on the script regexes, which are ranges rather than
proper `unicodedata` script lookups.*